## Step 1 — Read Bronze data

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_TABLE = "workspace.bronze.tracking_events"
TARGET_TABLE = "workspace.silver.tracking_events"

bronze_df = spark.table(SOURCE_TABLE)

print("Bronze records:", bronze_df.count())
bronze_df.printSchema()
display(bronze_df.limit(5))


## Step 2 — Convert and validate tracking events

In [0]:

VALID_EVENT_TYPES = [
    "PICKED_UP",
    "IN_TRANSIT",
    "AT_DISTRIBUTION_CENTER",
    "OUT_FOR_DELIVERY",
    "DELIVERED",
    "DELAYED",
]

transformed_df = (
    bronze_df
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.col("event_timestamp"),
            "yyyy-MM-dd'T'HH:mm:ssXXX"
        )
    )
)

valid_df = transformed_df.filter(
    F.col("event_id").isNotNull()
    & (F.trim(F.col("event_id")) != "")
    & F.col("shipment_id").isNotNull()
    & (F.trim(F.col("shipment_id")) != "")
    & F.col("event_timestamp").isNotNull()
    & F.col("event_type").isin(VALID_EVENT_TYPES)
    & F.col("distribution_center").isNotNull()
    & (F.trim(F.col("distribution_center")) != "")
    & F.col("_rescued_data").isNull()
)

print("Valid records:", valid_df.count())
print(
    "Rejected records:",
    transformed_df.count() - valid_df.count()
)


## Step 3 - Deduplicate by event ID

In [0]:

dedup_window = (
    Window
    .partitionBy("event_id")
    .orderBy(
        F.col("ingestion_timestamp").desc_nulls_last(),
        F.col("source_file").desc_nulls_last(),
        F.col("event_timestamp").desc_nulls_last()
    )
)

silver_df = (
    valid_df
    .withColumn(
        "row_number",
        F.row_number().over(dedup_window)
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number", "_rescued_data")
)

print("Valid records:", valid_df.count())
print("Silver records after deduplication:", silver_df.count())


## Step 4 — Write the Silver Delta table

In [0]:

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Silver table created: {TARGET_TABLE}")


### Validation

In [0]:
%sql
SELECT
    COUNT(*) AS total_events,
    COUNT(DISTINCT event_id) AS unique_events,
    COUNT_IF(event_timestamp IS NULL) AS invalid_timestamps
FROM workspace.silver.tracking_events;


### inspect event types:

In [0]:
%sql
SELECT
    event_type,
    COUNT(*) AS event_count
FROM workspace.silver.tracking_events
GROUP BY event_type
ORDER BY event_count DESC;

In [0]:
%sql
SELECT
    COUNT(*) AS total_events,
    COUNT(DISTINCT event_id) AS unique_events,
    COUNT_IF(event_timestamp IS NULL) AS invalid_timestamps
FROM workspace.silver.tracking_events;